In [ ]:
matching_results = "../results/matching_results_mstro_ollama:phi4.json"

In [5]:
import json
import sys
import os
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
import logging


def create_patient_report(patient_id, dataset_to_trials, results_folder):
    """
    Creates a detailed DOCX report for a single patient with all trials, 
    inclusion and exclusion criteria
    """
    # Create a subfolder for the patient
    patient_folder = os.path.join(results_folder, f"patient_{patient_id}")
    os.makedirs(patient_folder, exist_ok=True)
    
    # Create a new document
    doc = Document()
    
    # Add document title
    title = doc.add_heading(f"Patient: {patient_id}", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add document subtitle
    doc.add_paragraph(f"Clinical Trial Matching Report", style="Subtitle")
    doc.add_paragraph("").add_run("Generated by TrialGPT").italic = True
    
    # For each dataset
    for dataset_id, trials in dataset_to_trials.items():
        doc.add_heading(f"Dataset: {dataset_id}", level=1)
        
        # For each trial in this dataset
        for trial_id, trial_data in trials.items():
            doc.add_heading(f"Trial: {trial_id}", level=2)
            
            # Add summary section
            doc.add_heading("Summary", level=3)
            summary_table = doc.add_table(rows=1, cols=2)
            summary_table.style = 'Table Grid'
            header_cells = summary_table.rows[0].cells
            header_cells[0].text = "Category"
            header_cells[1].text = "Result"
            
            # Process inclusion criteria
            inclusion_count = {"included": 0, "not included": 0, "not applicable": 0, "not enough information": 0}
            if "inclusion" in trial_data:
                inclusion_data = trial_data["inclusion"]
                # Check if it's a dictionary (properly formatted) or a string (parse error)
                if isinstance(inclusion_data, dict):
                    for _, criterion_info in inclusion_data.items():
                        if isinstance(criterion_info, list) and len(criterion_info) >= 3:
                            status = criterion_info[2]
                            if status in inclusion_count:
                                inclusion_count[status] += 1
            
            # Process exclusion criteria
            exclusion_count = {"excluded": 0, "not excluded": 0, "not applicable": 0, "not enough information": 0}
            exclusion_error = False
            if "exclusion" in trial_data:
                exclusion_data = trial_data["exclusion"]
                # Check if it's a dictionary (properly formatted) or a string (parse error)
                if isinstance(exclusion_data, dict):
                    for _, criterion_info in exclusion_data.items():
                        if isinstance(criterion_info, list) and len(criterion_info) >= 3:
                            status = criterion_info[2]
                            if status in exclusion_count:
                                exclusion_count[status] += 1
                else:
                    exclusion_error = True
            
            # Add summary rows
            row = summary_table.add_row().cells
            row[0].text = "Inclusion Criteria"
            row[1].text = f"Included: {inclusion_count['included']}, Not included: {inclusion_count['not included']}, N/A: {inclusion_count['not applicable']}, Not enough info: {inclusion_count['not enough information']}"
            
            row = summary_table.add_row().cells
            row[0].text = "Exclusion Criteria"
            if exclusion_error:
                row[1].text = "PARSE ERROR - Could not process exclusion criteria"
                paragraph = row[1].paragraphs[0]
                run = paragraph.runs[0]
                run.font.color.rgb = RGBColor(255, 0, 0)
                run.bold = True
            else:
                row[1].text = f"Excluded: {exclusion_count['excluded']}, Not excluded: {exclusion_count['not excluded']}, N/A: {exclusion_count['not applicable']}, Not enough info: {exclusion_count['not enough information']}"
            
            # Detailed inclusion criteria
            if "inclusion" in trial_data and isinstance(trial_data["inclusion"], dict):
                doc.add_heading("Inclusion Criteria", level=3)
                inc_table = doc.add_table(rows=1, cols=3)
                inc_table.style = 'Table Grid'
                header_cells = inc_table.rows[0].cells
                header_cells[0].text = "Criterion ID"
                header_cells[1].text = "Explanation"
                header_cells[2].text = "Status"
                
                for criterion_id, criterion_info in trial_data["inclusion"].items():
                    if isinstance(criterion_info, list) and len(criterion_info) >= 3:
                        row = inc_table.add_row().cells
                        row[0].text = criterion_id
                        row[1].text = criterion_info[0]
                        row[2].text = criterion_info[2]
                        
                        # Color-code the status
                        if criterion_info[2] == "included":
                            row[2].paragraphs[0].runs[0].font.color.rgb = RGBColor(0, 128, 0)  # Green
                        elif criterion_info[2] == "not included":
                            row[2].paragraphs[0].runs[0].font.color.rgb = RGBColor(255, 0, 0)  # Red

            # Detailed exclusion criteria
            doc.add_heading("Exclusion Criteria", level=3)
            if "exclusion" in trial_data:
                if isinstance(trial_data["exclusion"], dict):
                    exc_table = doc.add_table(rows=1, cols=3)
                    exc_table.style = 'Table Grid'
                    header_cells = exc_table.rows[0].cells
                    header_cells[0].text = "Criterion ID"
                    header_cells[1].text = "Explanation"
                    header_cells[2].text = "Status"
                    
                    for criterion_id, criterion_info in trial_data["exclusion"].items():
                        if isinstance(criterion_info, list) and len(criterion_info) >= 3:
                            row = exc_table.add_row().cells
                            row[0].text = criterion_id
                            row[1].text = criterion_info[0]
                            row[2].text = criterion_info[2]
                            
                            # Color-code the status
                            if criterion_info[2] == "excluded":
                                row[2].paragraphs[0].runs[0].font.color.rgb = RGBColor(255, 0, 0)  # Red
                            elif criterion_info[2] == "not excluded":
                                row[2].paragraphs[0].runs[0].font.color.rgb = RGBColor(0, 128, 0)  # Green
                else:
                    p = doc.add_paragraph("ERROR: Could not parse exclusion criteria data. Raw content: ")
                    p.runs[0].font.color.rgb = RGBColor(255, 0, 0)
                    p.runs[0].bold = True
                    
                    # Add the raw exclusion data (limited to first 500 chars for readability)
                    raw_data = str(trial_data["exclusion"])
                    doc.add_paragraph(raw_data[:500] + ("..." if len(raw_data) > 500 else ""))
            else:
                doc.add_paragraph("No exclusion criteria data available.")
                
            # Add a page break between trials for better readability
            doc.add_page_break()
    
    # Save the document
    docx_path = os.path.join(patient_folder, f"patient_{patient_id}_detailed_report.docx")
    doc.save(docx_path)
    return docx_path

if __name__ == "__main__":
    # Path to the matching results
    matching_results_path = sys.argv[1]
    results_folder = "results"
    os.makedirs(results_folder, exist_ok=True)
    
    try:
        # Load the matching results
        with open(matching_results_path, 'r') as f:
            matching_results = json.load(f)
        
        # Process each patient
        for patient_id, dataset_to_trials in matching_results.items():
            try:
                docx_path = create_patient_report(patient_id, dataset_to_trials, results_folder)
                print(f"Created detailed report for patient {patient_id}: {docx_path}")
            except Exception as e:
                print(f"Error processing patient {patient_id}: {e}")
                traceback.print_exc()
    
    except Exception as e:
        print(f"Error loading matching results: {e}")
        traceback.print_exc()

Error loading matching results: [Errno 2] No such file or directory: '--f=/home/jovyan/.local/share/jupyter/runtime/kernel-v3515b537df0f18b273f2394a5e7c888fb1b66a9e2.json'


Traceback (most recent call last):
  File "/tmp/ipykernel_3652/235936023.py", line 161, in <module>
    with open(matching_results_path, 'r') as f:
  File "/home/jovyan/work/TrialGPT/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 324, in _modified_open
    return io_open(file, *args, **kwargs)
FileNotFoundError: [Errno 2] No such file or directory: '--f=/home/jovyan/.local/share/jupyter/runtime/kernel-v3515b537df0f18b273f2394a5e7c888fb1b66a9e2.json'


In [ ]:
!python rank_results.py /path/to/matching_results_mstro_ollama:phi4.json